# **Section 5: Bandwidth**

## **Introduction**
In this section, we will explore bandwidth usage in federated learning. We will measure and optimize the amount of data transferred between clients and the server to improve efficiency using the Flower framework.

---

## **Step 1: Import Required Libraries**


In [ ]:
pip install -r requirements.txt

In [ ]:
from flwr.client.mod import parameters_size_mod

from utils5 import *

---

## **Step 2: Define the Model**
We will initialize a pre-trained model from EleutherAI to measure its size and optimize bandwidth usage.

### **Load the Model**


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-14m",
    cache_dir="./pythia-14m/cache",
)

### **Measure Model Size**


In [ ]:
vals = model.state_dict().values()
total_size_bytes = sum(p.element_size() * p.numel() for p in vals)
total_size_mb = int(total_size_bytes / (1024**2))

log(INFO, "Model size is: {} MB".format(total_size_mb))

INFO : Model size is: 53 MB


---

## **Step 3: Implement Client-Side Communication**

### **Define Flower Client**


In [ ]:
class FlowerClient(NumPyClient):
    def __init__(self, net):
        self.net = net

    def fit(self, parameters, config):
        set_weights(self.net, parameters)
        # No actual training here
        return get_weights(self.net), 1, {}

    def evaluate(self, parameters, config):
        set_weights(self.net, parameters)
        # No actual evaluation here
        return float(0), int(1), {"accuracy": 0}

### **Create the Client Function and Client App**


In [ ]:
def client_fn(context: Context) -> FlowerClient:
    return FlowerClient(model).to_client()

client = ClientApp(
    client_fn,
    mods=[parameters_size_mod],  # Monitor bandwidth usage
)

---

## **Step 4: Implement Server-Side Strategy with Bandwidth Tracking**
We will track bandwidth usage by monitoring model parameter sizes during training.

### **Define Custom Strategy: BandwidthTrackingFedAvg**


In [ ]:
bandwidth_sizes = []

class BandwidthTrackingFedAvg(FedAvg):
    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        # Track sizes of models received
        for _, res in results:
            ndas = parameters_to_ndarrays(res.parameters)
            size = int(sum(n.nbytes for n in ndas) / (1024**2))
            log(INFO, f"Server receiving model size: {size} MB")
            bandwidth_sizes.append(size)

        # Call FedAvg for actual aggregation
        return super().aggregate_fit(server_round, results, failures)

    def configure_fit(self, server_round, parameters, client_manager):
        # Call FedAvg for actual configuration
        instructions = super().configure_fit(
            server_round, parameters, client_manager
        )

        # Track sizes of models to be sent
        for _, ins in instructions:
            ndas = parameters_to_ndarrays(ins.parameters)
            size = int(sum(n.nbytes for n in ndas) / (1024**2))
            log(INFO, f"Server sending model size: {size} MB")
            bandwidth_sizes.append(size)

        return instructions

### **Create Server Application**


In [ ]:
params = ndarrays_to_parameters(get_weights(model))

def server_fn(context: Context):
    strategy = BandwidthTrackingFedAvg(
        fraction_evaluate=0.0,
        initial_parameters=params,
    )
    config = ServerConfig(num_rounds=1)
    return ServerAppComponents(
        strategy=strategy,
        config=config,
    )

server = ServerApp(server_fn=server_fn)

---

## **Step 5: Run Federated Training with Bandwidth Tracking**

In [ ]:
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=2,
    backend_config=backend_setup,
)

INFO : Starting Flower ServerApp, config: num_rounds=1, no round_timeout
INFO : 
INFO : [INIT]
INFO : Using initial global parameters provided by strategy
INFO : Evaluating initial global parameters
INFO : 
INFO : [ROUND 1]
INFO : Server sending model size: 53 MB
INFO : Server sending model size: 53 MB
INFO : configure_fit: strategy sampled 2 clients (out of 2)
(pid=3016) 2025-02-08 22:28:25.892806: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=3016) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=3016) E0000 00:00:1739053705.941149    3016 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=3016) E0000 00:00:1739053705.955957    3016 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register facto

### **Log Bandwidth Usage**


In [ ]:
log(INFO, "Total bandwidth used: {} MB".format(sum(bandwidth_sizes)))


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
INFO : Total bandwidth used: 212 MB
